In [ ]:
import pandas as pd;
print(pd.__version__)

In [ ]:
import os, time, json, getpass, requests, pandas as pd
from pathlib import Path

# Make paths work from anywhere: after this, 'data/raw/...' always means <project>/data/raw/...
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
os.makedirs('data/raw', exist_ok=True)
print('Working in project folder:', ROOT.name)

# ListenBrainz requires a way to contact you. USE YOUR REAL EMAIL.
YOUR_EMAIL = getpass.getpass('Contact email for the User-Agent (hidden): ')
assert '@' in YOUR_EMAIL or YOUR_EMAIL.startswith('http'), 'Enter a real email or URL'

BASE = 'https://api.listenbrainz.org'
HEADERS = {'User-Agent': f'ListeningAnalysisProject/1.0 ( {YOUR_EMAIL} )'}

# Optional token (higher rate limits). In VS Code the prompt appears at the TOP of the window.
token = getpass.getpass('ListenBrainz token (press Enter to skip): ')
if token:
    HEADERS['Authorization'] = f'Token {token}'

def get_json(path, params=None):
    for attempt in range(5):
        r = requests.get(BASE + path, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429:                  # rate limited: wait as the server says
            wait = int(r.headers.get('X-RateLimit-Reset-In', 10))
            print(f'Rate limited, waiting {wait}s')
            time.sleep(wait + 1)
            continue
        if r.status_code == 404:                  # user not found
            return None
        r.raise_for_status()                      # any other error: stop and show it
        if int(r.headers.get('X-RateLimit-Remaining', 99)) <= 1:
            time.sleep(int(r.headers.get('X-RateLimit-Reset-In', 1)) + 1)
        time.sleep(1.1)                           # never more than one call per second
        return r.json()
    raise RuntimeError('Too many retries')

In [ ]:
good = Path('data/usernames.txt').read_text().split()
print(len(good), 'users loaded')

In [ ]:
def find_usernames(obj):
    """Collect every 'user_name' value, whatever shape the response has."""
    names = []
    if isinstance(obj, dict):
        if 'user_name' in obj:
            names.append(obj['user_name'])
        for v in obj.values():
            names += find_usernames(v)
    elif isinstance(obj, list):
        for v in obj:
            names += find_usernames(v)
    return names

terms = ['music', 'rock', 'jazz', 'indie', 'beats', 'dj']
candidates = []
for t in terms:
    data = get_json('/1/search/users/', {'search_term': t})
    found = find_usernames(data) if data else []
    print(t, '->', len(found), 'users')
    candidates += found

candidates = list(dict.fromkeys(candidates))   # remove duplicates, keep order
print(len(candidates), 'unique candidates')
print(candidates[:10])

In [ ]:
MIN_LISTENS = 3000
TARGET_USERS = 12

good = []
for i, u in enumerate(candidates):
    d = get_json(f'/1/user/{u}/listen-count')
    n = d['payload']['count'] if d else 0
    print(f'candidate {i}: {n} listens')          # prints the number, never the name
    if n >= MIN_LISTENS:
        good.append(u)
    if len(good) >= TARGET_USERS:
        break

print(len(good), 'users selected')
Path('data/usernames.txt').write_text('\n'.join(good))

In [ ]:
sample = get_json(f'/1/user/{good[0]}/listens', {'count': 3})
for l in sample['payload']['listens']:
    l = dict(l)
    l.pop('user_name', None)                  # keep the name out of the saved output
    print(json.dumps(l, indent=2)[:1500])
    print('-----')

In [ ]:
def fetch_listens(username, max_listens=5000, page_size=1000):
    rows, max_ts = [], None
    while len(rows) < max_listens:
        params = {'count': page_size}
        if max_ts:
            params['max_ts'] = max_ts            # only listens older than this
        data = get_json(f'/1/user/{username}/listens', params)
        if data is None:
            print('a user was not found')
            break
        listens = data['payload']['listens']
        if not listens:                           # nothing older: history exhausted
            break
        for l in listens:
            tm = l.get('track_metadata') or {}
            info = tm.get('additional_info') or {}
            rows.append({
                'user': username,
                'listened_at': l.get('listened_at'),
                'track': tm.get('track_name'),
                'artist': tm.get('artist_name'),
                'album': tm.get('release_name'),
                'duration_ms': info.get('duration_ms'),
                'music_service': info.get('music_service'),
                'client': info.get('submission_client'),
            })
        max_ts = listens[-1]['listened_at']       # the oldest listen on this page
        if len(listens) < page_size:              # a short page means the last page
            break
    return pd.DataFrame(rows).head(max_listens)

In [ ]:
test = fetch_listens(good[0], max_listens=2000)
print(test.shape)
print(test.isna().mean().round(2))
test.drop(columns='user').head()              # drop the username column before displaying

In [ ]:
frames = []
for i, u in enumerate(good):
    path = Path(f'data/raw/{u}.csv')
    if path.exists():                             # already downloaded: reuse it
        frames.append(pd.read_csv(path))
        print(f'user {i + 1}: loaded from cache')
        continue
    f = fetch_listens(u, max_listens=5000)
    f.to_csv(path, index=False)
    frames.append(f)
    print(f'user {i + 1}: {len(f)} listens downloaded')

raw = pd.concat(frames, ignore_index=True)
raw.to_csv('data/raw/listens_raw.csv', index=False)
print(raw.shape)

In [ ]:
labels = {u: f'user_{i + 1}' for i, u in enumerate(good)}

print(raw.isna().mean().round(3))
span = raw.groupby('user')['listened_at'].agg(['count', 'min', 'max'])
span['first_listen'] = pd.to_datetime(span['min'], unit='s')
span['last_listen'] = pd.to_datetime(span['max'], unit='s')
span.index = span.index.map(labels)              # anonymous labels for display
display(span[['count', 'first_listen', 'last_listen']])

print()
print(f"{len(raw):,} listens from {raw['user'].nunique()} users")
print('Earliest listen:', pd.to_datetime(raw['listened_at'].min(), unit='s'))
print('Latest listen:  ', pd.to_datetime(raw['listened_at'].max(), unit='s'))